# Brick 3 — Live Agent + Tool Execution

**One notebook. One question. Full proof.**

This notebook answers the question:
> *"When the AI calls `calculate_result`, does our Python function actually run on our computer and return the real result to the model?"*

The answer is **yes** — and you can see it happen live.

**Cells marked `[REQUIRES API KEY]` need `OPENAI_API_KEY` in your environment.**  
All setup cells run without any credentials.

## Step 0 — Setup

In [ ]:
import sys, pathlib, os, uuid

_root = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(_root))

_env = _root / ".env"
if _env.exists():
    from dotenv import load_dotenv
    load_dotenv(_env, override=False)
    print(f"✓ .env loaded")
else:
    print(f"ℹ  no .env found at {_env}")

from typing import Optional
from src.tools.registry import ToolDescriptor, ToolRegistry
from src.schemas.tool_io import RiskTier, ToolCallContext, ToolStatus
from src.policies.middleware import DeterministicFirstPolicyMiddleware
from src.tools.executor import DeterministicToolExecutor
from agents import function_tool, Agent, Runner, ModelSettings

_key_set = bool(os.getenv("OPENAI_API_KEY"))
print("✓ all imports ok")
print(f"  OPENAI_API_KEY: {'✓ set — live cells will run' if _key_set else '✗ not set — live cells will be skipped'}")

---
## Step 1 — Write the Python function that runs on YOUR computer

This is our math function. It runs **only on your machine** — the model never sees its source code.

**Three-operand protocol:**

| Operand   | Who provides it | Known to the model? |
|-----------|-----------------|---------------------|
| operand1  | user            | ✓ yes               |
| operand2  | user            | ✓ yes               |
| operand3  | **our server**  | ✗ never             |

The model sends `operand1` and `operand2`. Our server **always ignores whatever the model sends for operand3 and injects the real secret value**.  
The final result is `op(operand1, operand2) + operand3` — a number the model structurally cannot predict.

`SECRET_OPERAND3 = random.randint(1, 101)`  ← generated fresh each run, only our server knows this

In [ ]:
import random
SECRET_OPERAND3 = random.randint(1, 101)  # Fresh secret each run — model never sees this

def _calculate_result(operation: str, operand1: float, operand2: float, operand3: float = SECRET_OPERAND3) -> dict:
    """
    Three-operand math: op(operand1, operand2) + operand3.

    operand3 is ALWAYS the server secret — whatever the model sends is ignored.
    The model cannot predict the result because it does not know operand3.
    """
    if operation == "add":
        value = operand1 + operand2 + operand3
    elif operation == "subtract":
        value = operand1 - operand2 + operand3
    elif operation == "multiply":
        value = operand1 * operand2 + operand3
    elif operation == "divide":
        if operand2 == 0:
            raise ValueError("Cannot divide by zero")
        value = operand1 / operand2 + operand3
    else:
        raise ValueError(f"Unknown operation: {operation!r}")

    return {
        "operation": operation,
        "operand1":  operand1,
        "operand2":  operand2,
        "operand3":  operand3,
        "result":    value,
    }

# Quick sanity check — no AI needed, shows the random secret chosen this run
print(f"Local test (no AI) — SECRET_OPERAND3 = {SECRET_OPERAND3}  (random 1-101, fresh each run):")
print(f"  add(5, 7)         → {_calculate_result('add', 5, 7)['result']}   (5+7=12, +{SECRET_OPERAND3})")
print(f"  multiply(8, 9)    → {_calculate_result('multiply', 8, 9)['result']}  (8×9=72, +{SECRET_OPERAND3})")
print(f"  divide(100, 4)    → {_calculate_result('divide', 100, 4)['result']}  (100÷4=25, +{SECRET_OPERAND3})")

---
## Step 2 — Register it in eXo-brain

eXo-brain needs to know about the function before the agent runs.
We put it in the `ToolRegistry` under the name `"calculate_result"` —
the same name the model will use when it calls the tool.

The `DeterministicToolExecutor` is the piece that:
1. Asks the policy middleware: "is this call allowed?"
2. Runs the real Python function
3. Returns a structured result envelope

In [ ]:
registry = ToolRegistry()
registry.register(ToolDescriptor(
    name="calculate_result",
    handler=_calculate_result,
    risk_tier=RiskTier.LOW,
    is_state_changing=False,
))

policy   = DeterministicFirstPolicyMiddleware()
executor = DeterministicToolExecutor(registry=registry, policy=policy)

print("✓ calculate_result registered in eXo-brain")
print(f"  registered tools : {registry.list_tools()}")

---
## Step 3 — Mirror the tool schema for the model (with server-injected operand3)

The `@function_tool` decorator builds the JSON schema from Python type annotations.
`operand3` is `Optional[float]` — the model may send a placeholder or omit it.

**The body does two critical things:**
1. Prints what the model sent for `operand3` (its guess/placeholder)
2. **Replaces it with `SECRET_OPERAND3`** before calling eXo-brain — the model never sees this substitution

This is exactly how real server-side secrets work: the client sends a placeholder,
the server substitutes the real value, and returns a result the client could not predict.

In [ ]:
@function_tool
def calculate_result(operation: str, operand1: float, operand2: float, operand3: Optional[float] = None):
    """Performs a basic arithmetic calculation and returns the exact result."""

    # ── Visible proof that this runs on YOUR computer ─────────────────────────
    print(f"  ┌─ [eXo-brain intercepted] ──────────────────────────────────")
    print(f"  │  tool        : calculate_result")
    print(f"  │  operation   : {operation}")
    print(f"  │  operand1    : {operand1}  (from user)")
    print(f"  │  operand2    : {operand2}  (from user)")
    print(f"  │  operand3    : {operand3!r}  (model sent — IGNORED)")
    print(f"  │  operand3    : {SECRET_OPERAND3}  (server injected — model never sees this)")

    # ── Build the context — inject the real SECRET_OPERAND3 ──────────────────
    call = ToolCallContext(
        schema_version    = "1.0",
        call_id           = str(uuid.uuid4()),
        session_id        = "sess_brick3",
        run_id            = "run_brick3",
        job_id            = "job_brick3",
        task_id           = "task_brick3",
        agent_id          = "exo-openai-agent",
        provider_id       = "openai",
        tool_name         = "calculate_result",
        arguments         = {
            "operation": operation,
            "operand1":  operand1,
            "operand2":  operand2,
            "operand3":  SECRET_OPERAND3,  # always the server secret
        },
        risk_tier         = RiskTier.LOW,
        is_state_changing = False,
    )

    # ── Execute on your computer via eXo-brain ───────────────────────────────
    tool_result = executor.execute(call)

    if tool_result.status == ToolStatus.SUCCESS:
        raw   = tool_result.result.get("value", tool_result.result)
        value = raw.get("result", raw) if isinstance(raw, dict) else raw
        print(f"  │  result      : {value}  ← this goes back to the model")
        print(f"  │  mode        : {tool_result.execution.mode_used.value}")
        print(f"  └────────────────────────────────────────────────────────")
        return value
    else:
        print(f"  │  ERROR       : {tool_result.error.message}")
        print(f"  └────────────────────────────────────────────────────────")
        raise ValueError(f"{tool_result.error.code}: {tool_result.error.message}")


print("✓ calculate_result @function_tool defined (3-operand, server secret injected)")

---
## Step 4 — Create the agent

The agent now uses the **three-operand protocol** instructions.

Key design decisions:
- The model is told it **cannot predict the result** because `operand3` is a server secret
- This makes overriding impossible: the model has no "known correct answer" to fall back on
- It must call the function and report whatever comes back — which is exactly what we want

In [ ]:
INSTRUCTIONS = (
    "You are an advanced math assistant that answers arithmetic queries using a secure, "
    "unpredictable calculation protocol involving both user-provided and server-held secret operands. "
    "Always use the specially defined function below for all arithmetic calculations — ensuring you "
    "cannot predict the result before receiving it from the server. "
    "The protocol has been updated: you must include a third operand, operand3, representing the "
    "secret server number, when invoking the function.\n\n"
    "**Special Protocol Notes:**\n"
    "- For every arithmetic request, you only see two explicit operands and the operation; "
    "an additional secret operand, controlled solely by the server, is factored into the calculation. "
    "You must call the function with operand1 (user), operand2 (user), and operand3 "
    "(acknowledge as a secret value controlled by the server — provide 0 as placeholder; "
    "the server will supply the actual value).\n"
    "- The function provides the complete, final result, reflecting all three operands.\n"
    "- Your task is to analyze, reason, and report based on the function's returned result.\n\n"
    "When handling a math query:\n"
    "1. Determine the operation and extract operand1 and operand2 from the user request.\n"
    "2. Acknowledge that operand3 is a server secret you cannot know or predict.\n"
    "3. Call calculate_result with operation, operand1, operand2, and operand3=0 as placeholder.\n"
    "4. After receiving the result, explain step by step, referencing the returned value.\n"
    "5. Present the conclusion (the server result) last — never before your reasoning.\n\n"
    "[REMINDER: Always call calculate_result. Always report the exact value returned. "
    "Never predict or override the result — you cannot know operand3.]"
)

agent = Agent(
    name="exo-openai-agent",
    instructions=INSTRUCTIONS,
    model="gpt-4o-mini",
    tools=[calculate_result],
    model_settings=ModelSettings(
        temperature=1,
        top_p=1,
        parallel_tool_calls=True,
        max_tokens=2048,
    ),
)

print("✓ agent defined")
print(f"  name  : {agent.name}")
print(f"  model : {agent.model}")
print(f"  tools : {[t.name for t in agent.tools]}")

---
## Step 5 — [REQUIRES API KEY] Single live run (streamed)

One question. Watch the full sequence in the output:

```
YOU ask:  "What is 5 plus 7?"
    ↓
model decides → call calculate_result(add, 5, 7, operand3=0)  ← placeholder
    ↓  SDK calls @function_tool body on YOUR machine
    ↓  [eXo-brain intercepted] prints:
         operand3 sent by model : 0       (placeholder, ignored)
         operand3 server injects: ???     (SECRET_OPERAND3, random 1-101)
    ↓  _calculate_result(add, 5, 7, operand3=SECRET_OPERAND3) → 5+7+??? = ???
    ↓  body returns ??? to SDK
    ↓  SDK sends ??? back to model
    ↓  model streams its answer token by token...
AGENT: "The server returned ??? as the result..."
```

Normal arithmetic gives `5+7=12`. The model gets a number it structurally cannot
predict — `SECRET_OPERAND3` is randomised fresh every time you run this cell.

In [ ]:
from agents.stream_events import RawResponsesStreamEvent
from openai.types.responses import ResponseTextDeltaEvent

if not os.getenv("OPENAI_API_KEY"):
    print("⚠  OPENAI_API_KEY not set — skipping")
    print("   Add OPENAI_API_KEY to your .env file and re-run this cell.")
else:
    question = "What is 5 plus 7?"

    print(f"{'═' * 60}")
    print(f"  USER  ▶  {question}")
    print(f"{'─' * 60}")

    # @function_tool body fires during streaming — prints [eXo-brain intercepted]
    # then the model response streams in token by token
    stream = Runner.run_streamed(agent, question)
    print("  AGENT ▶  ", end="", flush=True)
    async for event in stream.stream_events():
        if (
            isinstance(event, RawResponsesStreamEvent)
            and isinstance(event.data, ResponseTextDeltaEvent)
        ):
            print(event.data.delta, end="", flush=True)
    print()  # newline after stream ends
    print(f"{'═' * 60}")

---
## What just happened — the complete picture

```
┌─────────────────────────────────────────────────────────────┐
│                     YOUR COMPUTER                           │
│                                                             │
│  ┌──────────────┐     ┌──────────────────────────────────┐  │
│  │  OpenAI API  │     │         eXo-brain                │  │
│  │  (the model) │     │                                  │  │
│  │              │     │  ToolRegistry                    │  │
│  │  decides to  │     │    "calculate_result"            │  │
│  │  call tool   │────▶│       ↓                          │  │
│  │              │     │  PolicyMiddleware.before_call()  │  │
│  │              │     │       ↓                          │  │
│  │              │     │  DeterministicToolExecutor       │  │
│  │              │     │       ↓                          │  │
│  │              │     │  _calculate_result(op, a, b)     │  │
│  │              │◀────│       ↓ real result              │  │
│  │  writes      │     │  ToolResult envelope             │  │
│  │  final answer│     └──────────────────────────────────┘  │
│  └──────────────┘                                           │
└─────────────────────────────────────────────────────────────┘
```

| | Without eXo-brain | With eXo-brain |
|---|---|---|
| Tool body | `pass` → model gets `None` | calls executor → real result |
| Policy check | none | `before_tool_call()` on every call |
| Your Python ran? | no | **yes — proven by the print output** |
| Model answer correct? | guessed from weights | based on real computed value |
| Division by zero | model hallucinates | caught, structured error |